# 06 · Line profiles and optical depth

This notebook goes from the optical-depth integral along a path to the sum that SkewerJAX evaluates, and checks every
ingredient of that sum. In order:

1. the derivation from the integral to the sum, with a two-element test;
2. the Voigt function;
3. the profiles at points;
4. closed-form checks;
5. the line-centred frames;
6. the linear detuning;
7. periodicity;
8. an application to simulation outputs.

Runtime: ~1 min.

**Requires:** lines (01), skewers (03, $a(x)$ from 02) and fields (04); the last section also simulation outputs (05).  
**Independent of:** the summary statistics (07).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'samplers', 'simulation', 'fields', 'tau',)); plt.show()

In [ ]:
from skewerjax.core import (LineBundle, LINES, voigt as V, line_wavelength_grid_A, frequency_offset_GHz, line_profiles,
                            compute_optical_depth, los_samples, resolution_report, wrap_to_ref_line, SkewerBundle, Fields,
                            ScalarField, VectorField, LogRamp, VelocityRamp, Constant, plane_points, analytic as an, phys)
lines = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"], ref_line_id="HI-LyA")
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")
from skewerjax.core import Cosmology

## 1 · From the integral to the sum

**The integral.** Along a straight comoving path, a photon is absorbed by transition $j$ of species $i$ with optical
depth (review notes, Eq. `tau_straight_los`, with $dl_{\rm phys} = a\, dx$)

$$
\tau_j(v_\dagger) = \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^3}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, a(x)\, dx
= \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^2}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, dx,
\qquad \mathcal{C}_j = \frac{e^2 f_j}{4\varepsilon_0 m_e c} .
$$

**The spectral coordinate.** The observer labels every photon by its line-centred velocity $v_\dagger$. At $x$, a gas
element with scale factor $a(x)$ and velocity $v_\parallel$ along the photon sees it at the local wavelength

$$
\lambda'_j = \lambda_j\, e^{v_\dagger/c}\, \frac{a(x)}{a_\dagger}\, D^{-1}(x),
\qquad D^{-1} = \sqrt{\frac{1+\beta}{1-\beta}},\qquad \beta = \frac{v_\parallel}{c}
$$

(review notes, Eq. `velocity_units_normalized`). The element absorbs at line centre ($\lambda'_j = \lambda_j$) when
$v_\dagger$ equals

$$
v_c(x) = c\left[\ln\frac{a_\dagger}{a(x)} + \frac12 \ln\frac{1-\beta}{1+\beta}\right] .
$$

Its frequency detuning is then an exact function of $v_\dagger - v_c$:

$$
\nu_j = c\left(\frac{1}{\lambda'_j} - \frac{1}{\lambda_j}\right) = \frac{c}{\lambda_j}\left[e^{-(v_\dagger - v_c)/c} - 1\right] .
$$

**Step 1: partition the path.** Point $i$ of a skewer stands for its dual segment $[m_{i-1/2}, m_{i+1/2}]$, of length
$w_i$, and the dual segments tile the path (03). A dual segment is cut into **pieces** $p$, one per cell it crosses,
of lengths $\ell_p$ with $\sum_{p \in i} \ell_p = w_i$. The integral is the sum of the integrals over the pieces.

- For simulation outputs the pieces are the exact intersections with the Voronoi cells (05).
- For fields sampled at the points there is one piece per point, with $\ell_p = w_i$ and the values at $x_i$.

**Step 2: what is constant within a piece.** These are constant on a piece:

- the gas of one cell: $n_{\rm com}$, $T$ (and so $\sigma$ and $\gamma$), and the velocity vector;
- $v_\parallel = \mathbf{v}\cdot\hat t$, because the path is straight within a piece.

What is **not** constant is $a(x)$. So $a^{-2}$ and the Hubble part of the line centre, $c\ln(a_\dagger/a(x))$, change
across the piece. Across a piece the line centre moves by

$$
W_p = \frac{dv_H}{dx}\, \ell_p,\qquad v_H(x) = c\ln\frac{a_\dagger}{a(x)} .
$$

**Step 3: one profile per piece.** The code evaluates $a$, $a^{-2}$ and $v_c$ at the centroid $s_p$ of every piece, the
midpoint rule on each piece:

$$
\tau_j(v_\dagger) \simeq \mathcal{C}_j \sum_p \frac{n_{{\rm com},p}\, \ell_p}{a_p^2}\, \phi_{j,p}(v_\dagger),
\qquad \phi_{j,p}(v_\dagger) = V\big(\nu_j(v_\dagger - v_{c,p});\, \sigma_p, \gamma_p\big),
$$

where:

- $\sigma_p = b_p/(\sqrt2\, \lambda_j)$, with $b_p = \sqrt{2k_BT_p/m} + b_{\rm turb}$;
- $\gamma_p = (A_{21} + A_{\rm coll})/(4\pi)$;
- $V$ is the Voigt function (§2).

This is `core.optical_depth_kernel`, fed by `los_samples` for fields at points and by
`simulations.absorbers.absorber_samples` for the pieces of a snapshot.

**Error, and when the sum is exact.**

- The midpoint rule replaces the profile smeared over $W_p$ by the profile at its centre. For a Gaussian core this
  changes the peak by a relative $W_p^2/(12 b^2)$: the sum converges to the integral at second order in the piece
  length.
- $\phi$ has unit area in $\nu$, so the column $\int \tau\, d\nu$ is exact for any partition.
- The sum equals the integral exactly when $a$ and $v_c$ are constant within every piece ($W_p = 0$). It converges to it
  as $\max_p W_p/b_p \to 0$.
- The resolution check of `compute_optical_depth` (`resolution_report`) monitors exactly this ratio. With the
  simulation default of 0.25 km/s spacing, $W_p \le 0.25$ km/s, far below any thermal width.

**Two elements, one $v_\dagger$.**

- The set-up:
  - element A at $x_r - d$, nearer the observer, recedes with $u_A > 0$;
  - element B at $x_r + d$ approaches with $u_B = -u_A$;
  - $u_A$ is chosen so that both line centres fall at the same $v_\dagger$.
- The two elements have different densities and temperatures. The integral, evaluated as a finely resolved
  quadrature, is compared with the code's sum for $K$ pieces per element.
- $\tau$ is additive along the path, and the sum is taken in real space, element by element. Each element therefore
  enters with its own $n$, $T$, $u$ and $a$, and two elements that absorb at the same $v_\dagger$ simply add their
  profiles. No mapping from $x$ to $v$ (and no Jacobian $dx/dv$) is ever formed, so several elements at one velocity
  are no special case.

In [ ]:
from skewerjax.utils import tutorial
from skewerjax.utils.plotting import plot_two_elements
r2 = tutorial.two_elements()          # x_ref = 10, d = 1.5, length 0.3 cMpc; Lya; K = 1, 4, 32 pieces per element
fig, axs = plot_two_elements(r2)
fig.suptitle("Two elements at the same distance from x = 10 cMpc, with opposite peculiar velocities: one v†, two profiles", fontsize=21)
plt.show()
print(f"u_A = {r2['u'][0]:+.1f} km/s, u_B = {r2['u'][1]:+.1f} km/s; both absorb at v† = {r2['v_star']:.1f} km/s; a_A = {r2['a'][0]:.5f}, a_B = {r2['a'][1]:.5f}")
print(f"b_A = {r2['b_kms'][0]:.1f}, b_B = {r2['b_kms'][1]:.1f} km/s; Hubble width of one element W = {r2['hubble_width_kms']:.1f} km/s")
for K in sorted(r2["tau_sum"]):
    print(f"K = {K:2d}: max |sum - integral| / tau_max = {np.abs(r2['tau_sum'][K] - r2['tau_ref']).max() / r2['tau_ref'].max():.2e}, "
          f"column (integral of tau dv) error {abs(r2['tau_sum'][K].sum() / r2['tau_ref'].sum() - 1):.1e}")
print(f"sum of the two per-element sums - the sum: {np.abs(r2['tau_sum_parts'][32].sum(0) - r2['tau_sum'][32]).max():.1e}")

*Reading the figure.*

- (a) The grey line is where gas at rest would absorb. The arrows are the peculiar velocities, which move both elements
  to the same line centre.
- (b) Each element alone (dashed) has its own width: A is cold and narrow, B is hot and broad. Their sum is the total,
  on top of the reference integral.
- (c) The error of the sum falls as $K^{-2}$, i.e. as $(W/b)^2$. With one piece per element ($W/b_A \approx 1.9$) the
  Hubble smearing inside each element is missed. With 32 pieces the error is $\sim 10^{-4}$ of $\tau_{\max}$.

## 2 · The Voigt function

Three implementations of the same function $V(x; \sigma, \gamma) = (G_\sigma * L_\gamma)(x)$, the convolution of a
Gaussian of standard deviation $\sigma$ and a Lorentzian of half-width $\gamma$:

- `"ExoJAX"` (default): a Humlíček-type approximation of the Faddeeva function, compatible with `jit` and `grad`;
- `"convolution"`: $G * L$ by FFT on the grid, refined 4 times; it needs a grid that resolves both widths;
- `"scipy_wofz"`: the reference, in NumPy, with

$$
V = \frac{\mathrm{Re}\, w(z)}{\sqrt{2\pi}\, \sigma},\qquad z = \frac{x + i\gamma}{\sqrt2\, \sigma} .
$$

The four $(\sigma, \gamma)$ pairs combine $10^{-3}$ and $1$:

- the Gaussian limit ($\gamma \ll \sigma$);
- the Lorentzian limit ($\sigma \ll \gamma$);
- two Voigt profiles with both widths equal.

In [ ]:
from skewerjax.utils.plotting import plot_voigt_limits
x = jnp.linspace(-9, 9, 2 ** 13)
sigma = np.array([1e-3, 1.0, 1e-3, 1.0]); gamma = np.array([1e-3, 1e-3, 1.0, 1.0])
curves = {"scipy_wofz": np.stack([V.voigt_scipy(np.asarray(x), s, g) for s, g in zip(sigma, gamma)]),
          "ExoJAX": np.stack([np.asarray(V.voigt_exojax(x, s, g)) for s, g in zip(sigma, gamma)]),
          "convolution (FFT, oversample 4)": np.stack([np.asarray(V.voigt_convolution(x, s, g, oversample=4)) for s, g in zip(sigma, gamma)])}
fig, axs = plot_voigt_limits(np.asarray(x), sigma, gamma, curves)
add_panel_letters(axs); fig.suptitle("Voigt: the Gaussian and Lorentzian limits; ExoJAX (solid) and FFT convolution (dashed) on scipy's wofz", fontsize=21)
plt.show()
for name in ("ExoJAX", "convolution (FFT, oversample 4)"):
    r = np.abs(curves[name] / curves["scipy_wofz"] - 1)
    core = np.abs(np.asarray(x))[None] < 3 * np.maximum(sigma, gamma)[:, None]
    print(f"{name:32s} max |V / wofz - 1| per pair (|x| < 3 max(sigma, gamma)):", np.array2string(np.where(core, r, 0).max(1), precision=1))

ExoJAX agrees with `wofz` to $\sim 10^{-6}$ or better for every pair. The convolution is exact ($10^{-14}$) when the
grid resolves both widths, and it fails where the grid cannot:

- $\sigma = \gamma = 10^{-3}$ is far below the grid spacing ($2.2\times10^{-3}$);
- the Lorentzian wings are cut at the grid ends.

Full checks: `scripts/validation/check_voigt.py`.

## 3 · Line profiles at points

`line_profiles(events, directions, v, a_†, lines, fields)` evaluates $\phi_{ij}$ for every line and every point.

- A hotter gas broadens the lines.
- A peculiar velocity along the photon shifts them: $v_c \approx c\ln(a_\dagger/a) - v_\parallel$, where $v_\parallel$
  is measured along the photon direction. Gas moving with the photon, i.e. towards the observer, is blueshifted to
  smaller $v_\dagger$.
- In panel (b) the photons travel along $+x$, so the points with $v_x > 0$ absorb at negative $v_\dagger$.

In [ ]:
pts = plane_points(1e-3, "y", 5e-4, 1, 5)[0]
events = jnp.concatenate([jnp.ones((5, 1)), pts], -1)
v = jnp.linspace(-400, 400, 801)
fig, axs = plt.subplots(1, 2, figsize=(20, 6), layout="constrained")
for ax, (Tlo, Thi, vlo, vhi, title) in zip(axs, ((1e3, 1e6, 0.0, 0.0, r"$T$ from $10^3$ to $10^6$ K"), (1e4, 1e4, -300.0, 300.0, r"$v_x$ from −300 to +300 km/s, $T = 10^4$ K"))):
    sp = {"temperature": ScalarField(LogRamp(Tlo, Thi, 1e-3, 3)), "velocity": VectorField(VelocityRamp(vlo, vhi, 1e-3)),
          "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0)), "number_density": ScalarField(Constant(1.0))}
    ev = events.at[:, 3].set(jnp.linspace(0, 1e-3, 5)) if Tlo != Thi else events.at[:, 1].set(jnp.linspace(0, 1e-3, 5))
    phi = np.asarray(line_profiles(ev, jnp.broadcast_to(jnp.array([1.0, 0, 0]), (5, 3)), v, 1.0, LineBundle(["HI-LyA"]), Fields({"HI": sp})))[0]
    for i in range(5):
        ax.semilogy(v, phi[i], color=list(TOL.values())[i])
    ax.set(xlabel=r"$v_\dagger$ [km/s]", ylabel=r"$\phi$ [ns]", ylim=(1e-8, None), title=title)
add_panel_letters(axs); fig.suptitle(r"Ly$\alpha$ profiles: temperature sets the width, the line-of-sight velocity the centre", fontsize=22); plt.show()

The detuning $\nu_j$ of §1 uses the exact exponential mapping by default; `detuning="linear"` replaces it by its first
order (§6). `velocity_period=P` replaces $v_\dagger - v_c$ by its nearest image in $[-P/2, P/2)$ (§7).

## 4 · Optical depth: checks against closed forms

Static uniform gas ($a = 1$, no velocities) has closed forms (review notes):

- a Lorentzian for $T = 0$;
- a Gaussian for $A_{21} = 0$, the `null...` lines of the table.

Lines of one species share the fields.

In [ ]:
Lk = 1e-3
def fields(T, n):
    sp = lambda: {"temperature": ScalarField(LogRamp(*T, Lk, 3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lk)),
                  "number_density": ScalarField(LogRamp(*n, Lk, 2)), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
    return Fields({"HI": sp(), "SiIII": sp()})
null = LineBundle(["nullHI-LyA", "nullHI-LyB", "nullSiIII1206"])
pts = plane_points(Lk, "y", Lk / 2, 4, 32)                                       # 4 skewers, temperature rising with z
sb = SkewerBundle(spatial_points=pts, manual_scale_factors=jnp.ones_like(pts[None, ..., 0]))
v = jnp.linspace(-3000.0, 3000.0, 1024)
F_th = fields((1e4, 1e8), (1e-7, 1e-7))
tau = compute_optical_depth(sb, v, null, F_th, a_dagger=1.0)
T_sk = F_th["HI"]["temperature"](sb.quadri_vectors[0])[:, 0]
wl = line_wavelength_grid_A(v, null, 1.0, 1.0)
ref = jnp.stack([an.tau_static_thermal(wl[j], null.lambda0_A[j:j+1], null.mass_amu[j:j+1], null.f_osc[j:j+1], sb.a0, Lk * jnp.ones(4),
                                       1e-7 * jnp.ones(4), T_sk)[0] for j in range(3)])
F, Fa = np.exp(-np.asarray(tau)), np.exp(-np.asarray(ref))
fig, axs = plt.subplots(2, 3, figsize=(22, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
for j, l in enumerate(lines.lines):
    for k in range(4):
        c = list(TOL.values())[k]
        axs[0, j].plot(v, Fa[j, k], color=c, **MS["reference"])
        axs[0, j].plot(v, F[j, k], color=c, **MS["default"], label=f"T = {float(T_sk[k]):.0e} K" if j == 0 else None)
        axs[1, j].plot(v, F[j, k] - Fa[j, k], color=c)
    axs[0, j].set_title(l.name); axs[1, j].set_xlabel(r"$v_\dagger$ [km/s]")
axs[0, 0].set_ylabel("F"); axs[1, 0].set_ylabel(r"$F - F_{\rm analytic}$"); axs[0, 0].legend(fontsize=15)
add_panel_letters(axs); fig.suptitle("Static thermal lines: SkewerJAX (solid) on the analytic Gaussians (light); residuals < 10⁻⁸", fontsize=22); plt.show()

Radiation-dominated expansion and the convergence with the sampling are checked in
`scripts/validation/check_optical_depth_analytic.py`.

*Reading the figure.*

- (a) On the observed-wavelength axis, each line absorbs between two wavelengths:
  - its rest wavelength $\lambda_0$ (dash-dot), at the end of the path at $a_0$;
  - $\lambda_0 a_0/a_{\rm far}$ (dashed), where the light left the other end of the box.

  The Hubble flow across 5 cMpc spreads the absorption over a few hundred km/s, to the red or to the blue of
  $\lambda_0$, depending on where the initial conditions sit.
- (b–d) Around each line's $v_\dagger = 0$, the numerical $F$ lies on the closed form.
- (e–g) The residuals stay inside the $\pm 10^{-2}$ band and fall as the square of the point spacing (§1).

## 5 · Line-centred frames, the global velocity axis and the observed wavelength

Every line $j$ is integrated on the same grid of line-centred velocities $v_\dagger$, its own frame. By §1, the line
centre of the gas at $x$ is the same $v_c(x)$ in every frame: it does not depend on $\lambda_j$
(`core.spectral.line_centre_velocity_kms`).

The observer sees one axis, and there are two ways to write it:

- **Observed wavelength** (`spectral.line_wavelength_grid_A`):

$$
\lambda_{\rm obs} = \lambda_j\, \frac{a_{\rm obs}}{a_\dagger}\, e^{v_\dagger/c} .
$$

- **The global velocity axis** of a reference line (here Ly$\alpha$): the same pixel of line $j$ sits at
  $v_{\rm Ly\alpha} = v_\dagger - \Delta v_j$, with

$$
\Delta v_j = c\ln\frac{\lambda_{\rm Ly\alpha}}{\lambda_j}
$$

  (`LineBundle.delta_v`). `wrap_to_ref_line` performs that shift on the uniform grid, treating the grid as periodic (§7).

**The gas.** Four clumps along a 20 cMpc skewer at $z \approx 3$, with no peculiar velocities. Each clump has its own
width, peak density and temperature ($8\times10^3$ to $6\times10^4$ K), so that every absorber is easy to tell apart.
The lines are Ly$\alpha$ and Si III $\lambda$1206, with Si III 5 times less abundant to make it visible.

In [ ]:
from skewerjax.core import Cosmology, SampledField
from skewerjax.utils.plotting import plot_frames_axes
eds = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
L20, n_pt, z_obs = 20.0, 8193, 3.0
xs = np.linspace(0.0, L20, n_pt)
pts = jnp.asarray(np.stack([xs, np.full_like(xs, 1.0), np.full_like(xs, 1.0)], -1)[None])
sbf = SkewerBundle(spatial_points=pts, cosmologies=eds, a0=1 / (1 + z_obs), x0=L20)      # observer beyond the far end
# four clumps (position [cMpc], width [cMpc], peak density [cm^-3], temperature [K]): each one different
clumps = [(3.0, 0.12, 1.2e-12, 8e3), (7.5, 0.25, 0.4e-12, 6e4), (12.0, 0.10, 2.0e-12, 1.5e4), (16.5, 0.20, 0.8e-12, 3e4)]
g = np.stack([n0 * np.exp(-0.5 * ((xs - c0) / w0) ** 2) for c0, w0, n0, _ in clumps])
n_x = g.sum(0) + 1e-14
T_x = (g * np.array([c[3] for c in clumps])[:, None]).sum(0) / np.maximum(g.sum(0), 1e-300) * (g.sum(0) > 1e-16) + 1e4 * (g.sum(0) <= 1e-16)
sp_c = lambda f: {"temperature": ScalarField(SampledField(jnp.asarray(T_x)[None])), "velocity": VectorField(VelocityRamp(0.0, 0.0, L20)),
                  "number_density": ScalarField(SampledField(jnp.asarray(f * n_x)[None])), "turbulence": ScalarField(Constant(0.0)),
                  "collisional": ScalarField(Constant(0.0))}
lya_si = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
a_pts = np.asarray(sbf.scale_factors[0, 0])
a_dag = 1 / (1 + z_obs)
P_box = float(phys.c_kms * np.log(a_pts.max() / a_pts.min()))                          # the path in velocity
v_box = (np.arange(2048) + 0.5) * P_box / 2048 + float(phys.c_kms * np.log(a_dag / a_pts.max()))
tau_f = compute_optical_depth(sbf, v_box, lya_si, Fields({"HI": sp_c(1.0), "SiIII": sp_c(1 / 5)}), a_dagger=a_dag)
v_cl = [float(phys.c_kms * np.log(a_dag / np.interp(c0, xs, a_pts))) for c0, *_ in clumps]
fig, axs = plot_frames_axes(v_box, np.exp(-np.asarray(tau_f)[:, 0]), lya_si, a_dagger=a_dag, a_obs=1.0, centres=v_cl)
add_panel_letters(axs)
fig.suptitle("The same four clumps in observed wavelength (a), on the global Lyα axis (b) and in each line's own frame (c)", fontsize=21)
plt.show()
print(f"path: {P_box:.1f} km/s of Hubble flow; Delta v(Si III) = {float(lya_si.delta_v[1]):.1f} km/s = "
      f"{float(lya_si.delta_v[1]) / P_box:+.3f} periods; clump centres v_c = {np.round(v_cl, 1)} km/s (the same in both own frames)")
print("clumps (x, width, n, T):", [(c0, w0, f"{n0:.1e}", f"{T0:.0e}") for c0, w0, n0, T0 in clumps])

*Reading the figure.*

- (c) In its own frame each line absorbs at the clump's $v_c$ (dotted): the same gas at the same velocity, in every
  line.
  - Each line's width is the clump's extent in Hubble flow plus its thermal width. The compact clumps make narrow lines,
    and the hot, extended one a broad, shallow line.
  - Si III is only slightly narrower than Ly$\alpha$: its thermal width is 5.3 times smaller (Si is 28 times heavier),
    but the Hubble extent is the same for both lines.
- (a) On the observed-wavelength axis the two lines are $9.2\ \text{Å} \times (a_{\rm obs}/a_\dagger)$ apart.
- (b) On the Ly$\alpha$ frame, Si III moves by $-\Delta v_j = -2271$ km/s, to the blue.
  - `wrap_to_ref_line` (solid) brings what leaves the grid back in at the other end, as in a periodic box.
  - The unwrapped shift (dashed) shows where the same absorption would sit on an axis without a period.
  - The shift is exact only when $\tau$ itself is periodic on the grid (§7).

## 6 · The linear detuning

The exact detuning of §1, $\nu_j = (c/\lambda_j)[e^{-(v_\dagger - v_c)/c} - 1]$, can be replaced by its first order,

$$
\nu_j^{\rm lin} = -\frac{c}{\lambda_j}\, \frac{v_\dagger - v_c}{c},\qquad
\frac{\nu_j^{\rm lin}}{\nu_j} = 1 + \frac{v_\dagger - v_c}{2c} + \mathcal{O}\left(\frac{(v_\dagger - v_c)^2}{c^2}\right)
$$

(`detuning="linear"`). **What it approximates:** only the map from the velocity offset to the frequency offset. The line
centres $v_c$ and the widths are the same in both.

**When it matters.**

- The relative change of $\nu$ is $(v_\dagger - v_c)/(2c)$: $7\times10^{-5}$ at 40 km/s and $1.7\times10^{-3}$ at
  1000 km/s.
- In a Gaussian core $\tau$ is significant only within a few $b$ of the centre, where the change is negligible.
- In Lorentzian wings $\phi \propto \gamma/\nu^2$, so $\tau^{\rm lin}/\tau - 1 \approx -(v_\dagger - v_c)/c$. This is
  antisymmetric: the linear map makes the two wings symmetric.
- The relative error does not depend on the column, because $\tau \propto N$. The flux error peaks where $\tau \sim 1$
  in the wings, which only strong systems reach far from the centre.

The test: one Ly$\alpha$ absorber at rest with $\log_{10} N = 14$ (forest), 17 (Lyman limit) and 20.5 (damped).

In [ ]:
from skewerjax.utils.plotting import plot_linear_detuning
dl = tutorial.linear_detuning(log_N=(14.0, 17.0, 20.5), T=1.5e4)      # one Lya absorber at rest, three columns
fig, axs = plot_linear_detuning(dl)
add_panel_letters(axs); fig.suptitle("The linear detuning: a relative error of order (v† − v_c)/c, visible only in damping wings", fontsize=21)
plt.show()
rel = dl["tau_linear"][0] / dl["tau_exact"][0] - 1
for dv_ in (40.0, 1000.0, 2000.0):
    print(f"v - v_c = {dv_:+6.0f} km/s: tau_lin / tau_exact - 1 = {np.interp(dv_, dl['v'], rel):+.2e} (wing estimate -dv/c = {-dv_ / phys.c_kms:+.2e})")
print("max |F_lin - F_exact| per column:", {f"logN={n:.1f}": f"{x:.1e}" for n, x in zip(dl["log_N"], np.abs(np.exp(-dl["tau_linear"]) - np.exp(-dl["tau_exact"])).max(1))})

*Reading the figure.* (b) follows $-(v_\dagger - v_c)/c$ in the wings. Near the centre, the Gaussian core changes sign
and size, but $\tau$ is tiny there in relative terms. (c) The flux changes by at most $\sim 10^{-3}$, and only in the
damping wings of the strongest absorber. For forest absorbers the change is below $10^{-4}$ everywhere.

## 7 · Periodicity: what the code does, why it matters, and the options

**Why it matters.** A simulation box is periodic: the gas past its far face is the gas at its near face. A skewer that
crosses the box once spans one period in velocity, $P = c\ln(a_0/a_L)$, and its spectrum is naturally periodic with
that period. Two things reach past the ends of the grid:

- the profile of gas within a few $b$ of either face;
- every line shifted onto the frame of another line by its offset $\Delta v_j$, which is larger than $P$ for Si III on
  Ly$\alpha$ in small boxes.

**What the code does, and where.**

- **The sum of §1 is not periodic.** In the core, `compute_optical_depth(..., velocity_period=None)` (the default)
  evaluates it as it is, for any path and any grid.
- **`velocity_period=P`** (core, opt-in, in `spectral.frequency_offset_GHz`) replaces every detuning
  $v_\dagger - v_c$ by its nearest image in $[-P/2, P/2)$. This is the one place where periodicity enters $\tau$.
- **The simulation layer chooses it by default.** `IntegrationOptions(periodic=True)` gives the frame the period of the
  box, and `SimulationSkewers.spectral_grid` covers exactly one period. The frame passes `velocity_period` to the core.
- **`wrap_to_ref_line`** (core) shifts each line onto the reference line's frame, and always treats the grid as one
  period, $N_v\, dv$.
- **The gas itself** is periodic in space wherever it comes from a periodic box. For toy fields, `GRFDensity`
  interpolates in the periodic box. For a snapshot, the selection and the deposition take every cell at its minimum
  image, so paths may cross the box faces. This is the topology of the gas, not an option of $\tau$.

**The options, step by step**, on one absorber 0.25 cMpc from the far face of a 20 cMpc box:

- (a) **No wrap.** The grid $[0, P)$ receives what lands on it, and the part of the profile beyond $P$ is lost.
- (b) **Detunings wrapped modulo $P$** (`velocity_period=P`). The part beyond $P$ re-enters at the near end of the grid.
- (c) **A buffer.** The path is extended at both ends through the periodic gas (`SimulationSkewers.axis_aligned(...,
  extend_cMpc=...)`). The integral stays non-periodic, and the spectrum is read on $[0, P)$.
- (d) **`wrap_to_ref_line`**, a post-processing shift. Si III's absorption moves by $-\Delta v$ modulo $P$.

In [ ]:
from skewerjax.utils.plotting import plot_periodicity_options
pb = tutorial.periodic_box_demo()      # a 20 cMpc periodic box at z = 3; one Lya absorber 0.25 cMpc from the far face
fig, axs = plot_periodicity_options(pb)
add_panel_letters(axs); fig.suptitle("Periodicity, step by step: (a) no wrap, (b) wrapped detunings, (c) a buffer, (d) one axis for two lines", fontsize=21)
plt.show()
t = pb["tau"]
print(f"P = {pb['period_kms']:.1f} km/s; the absorber's line centre at {pb['v_c']:.1f} km/s, {pb['period_kms'] - pb['v_c']:.1f} km/s from the end of the grid")
print(f"integral of tau over the grid: no wrap {t['none'].sum():.2f}, wrapped {t['wrapped'].sum():.2f}, buffer {t['buffer'].sum():.2f} "
      f"(pixels x tau); max |wrapped - buffer| / tau_max = {np.abs(t['wrapped'] - t['buffer']).max() / t['wrapped'].max():.1e}")

**The same options on a toy box.** Four skewers through the 50 cMpc box of notebook 00 ($z = 2.2$), with one
`PowerLawGRF` for H I and Si III ($n = \bar n (1 + \delta)$), on a grid of exactly one period:

- (a) No wrap differs from the wrapped detunings only near the ends of the grid, where gas near the faces spills over.
- (b) On the wrapped $\tau$ the shift onto the Ly$\alpha$ frame is exact. Si III moved by `wrap_to_ref_line` equals
  Si III evaluated directly at $v + \Delta v$ in its own frame, up to the linear interpolation between pixels.
- (c) Shifting the unwrapped $\tau$ moves its cut edge to the middle of the reference-frame spectrum (dotted:
  $-\Delta v$ mod $P$).

In [ ]:
from skewerjax.core import grid_skewer_points, PowerLawGRF, GRFDensity
eds = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
Lb, z = 50.0, 2.2
pts = grid_skewer_points(Lb, 2, 16385, direction="y").reshape(4, 16385, 3).astype(jnp.float64)   # first and last point: the same place
sb4 = SkewerBundle(spatial_points=pts, cosmologies=eds, a0=1 / (1 + z), x0=Lb)
a_d = 1 / (1 + z)
P = float(phys.c_kms * jnp.log(a_d / sb4.scale_factors[0, 0, 0]))             # the box in velocity: one period [km/s]
vq = (np.arange(2048) + 0.5) * P / 2048                                         # one period, N_v dv = P exactly
grf_q = PowerLawGRF(A_s=0.04, k_0=2.0, n_s=-1.25, L=(Lb,) * 3, N=(64,) * 3)     # sigma_delta = 0.20
key_q = jax.random.PRNGKey(0)
dens_HI = GRFDensity(key_q, grf_q, n_bar=1e-12)                                # n = n_bar (1 + delta)
dens_q = {"HI": dens_HI, "SiIII": GRFDensity(None, grf_q, n_bar=1e-12 / 50, delta_grid=dens_HI.delta_grid)}
print(f"sigma_delta = {float(jnp.sqrt(grf_q.variance())):.3f}, min(1 + delta) = {float(dens_HI.min_one_plus_delta):.3f}")
toy_sp = lambda d: {"temperature": ScalarField(LogRamp(1.5e4, 1.5e4, Lb)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lb)),
                    "number_density": ScalarField(d), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
toy = Fields({sp: toy_sp(d) for sp, d in dens_q.items()})
dV = float(lya_si.delta_v[1])
tau_per = compute_optical_depth(sb4, vq, lya_si, toy, a_dagger=a_d, velocity_period=P)          # wrapped detunings
tau_dir = compute_optical_depth(sb4, vq, lya_si, toy, a_dagger=a_d)                             # no wrap (default)
si_only = LineBundle(["SiIII-1206"], ref_line_id="SiIII-1206")
tau_si_shift = compute_optical_depth(sb4, vq + dV, si_only, Fields({"SiIII": toy_sp(dens_q["SiIII"])}), a_dagger=a_d, velocity_period=P)[0]
F_per, F_dir = np.exp(-np.asarray(tau_per)), np.exp(-np.asarray(tau_dir))
F_ref_per = np.exp(-np.asarray(wrap_to_ref_line(tau_per, vq, lya_si)))
F_ref_dir = np.exp(-np.asarray(wrap_to_ref_line(tau_dir, vq, lya_si)))
fig, axs = plt.subplots(1, 3, figsize=FIGSIZES["triple"], layout="constrained")
for k in range(4):
    c = list(TOL.values())[k]
    axs[0].plot(vq, F_dir[0, k] - F_per[0, k], color=c, label=f"skewer {k}")
    axs[1].plot(vq, F_ref_per[1, k] - np.exp(-np.asarray(tau_si_shift[k])), color=c)
    axs[2].plot(vq, F_ref_dir[1, k] - F_ref_per[1, k], color=c)
axs[0].set(xlabel=LABELS["v_dagger"], ylabel=r"$F_{\rm no\,wrap} - F_{\rm wrapped}$ (Ly$\alpha$)", title="(a) no wrap vs wrapped detunings")
axs[1].set(xlabel=LABELS["v_ref"], ylabel=r"$\Delta F$ (Si III)", title=r"(b) shift vs direct evaluation at $v + \Delta v$")
axs[2].set(xlabel=LABELS["v_ref"], ylabel=r"$\Delta F$ (Si III)", title="(c) shift of the unwrapped τ − shift of the wrapped one")
axs[2].axvline(np.mod(-dV, P), color=FG, ls=":", lw=1.2)
axs[0].legend(); add_panel_letters(axs); plt.show()
print(f"P = {P:.1f} km/s | max |dF|: (a) {np.abs(F_dir - F_per).max():.2e} | (b) {np.abs(F_ref_per[1] - np.exp(-np.asarray(tau_si_shift))).max():.2e} "
      f"| (c) {np.abs(F_ref_dir[1] - F_ref_per[1]).max():.2e}")

**Trade-offs.**

- **No wrap** is the right integral for a path that is not a box: a light cone, a curved path, a segment of a longer
  sightline.
  - It is the cheapest, and every element absorbs where it physically is.
  - In a periodic box it loses the absorption of the gas near the faces (a few $b$ worth), which biases the mean flux
    near the edges and the largest scales of P1D.
- **Wrapped detunings** are exact for a periodic box when the grid is one period and the skewer crosses the box once,
  closing on itself.
  - They need no extra path or gas.
  - Their only approximation is that the wrapped gas keeps the scale factor of the face it belongs to, rather than that
    of a periodic copy beyond the other face. That is a difference of order $P/c$ in $a$, so $\sim 2P/c \approx 1\%$
    in $a^{-2}$ for this box.
- **A buffer** is explicit and general: any grid, any range, a non-periodic integral.
  - It costs extra path, extra selected gas and a longer integral.
  - On one period it agrees with the wrapped detunings to $4\times10^{-3}$ of $\tau_{\max}$ in (c) above. The difference is
    the scale factor of the copy, just described.
- **`wrap_to_ref_line`** is a post-processing step, so it can be applied to any $\tau$.
  - It is exact only if $\tau$ is periodic on the grid, with period $N_v\, dv$.
  - It interpolates linearly between pixels.
  - Applied to a non-periodic $\tau$, it brings the cut edge into the middle of the spectrum.

## 8 · Application: $\tau$ from simulation outputs

The same integrator on the gas of notebook 05. `SimulationSkewers.optical_depth(lines)` integrates every line in its
own frame over one period of the box (1 km/s pixels by default), from the deposited pieces of every dual segment (§1).
`fields(species)` gives the same gas as a `core.Fields` at the points.

In [ ]:
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
from skewerjax.core import Cosmology
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, IntegrationOptions, HI
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
cosmo = Cosmology(H0=100 * H.h * 1e6 * phys.Gyr_to_s / phys.Gpc_to_m, Omega_r_0=0.0, Omega_m_0=H.omega_m, Omega_k_0=0.0, Omega_lambda_0=H.omega_l)
L = H.box_cMpc
x = np.linspace(0.0, L, int(round(H.vmax_kms / 0.25)) + 1)
points = np.stack([np.stack([x, np.full_like(x, y), np.full_like(x, z)], -1) for y, z in [(10.0, 30.0), (25.0, 5.0), (40.0, 45.0)]])
bundle = SkewerBundle(spatial_points=jnp.asarray(points), a0=H.a, ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
sk = SimulationSkewers.from_bundle(reader, bundle)
print(f"TNG50-4 snapshot 25: the three skewers of notebook 05, {len(sk.cells):,} cells selected in {sk.selection_s:.1f} s")

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
velocities, tau = sk.optical_depth(lines)
flux = np.exp(-tau)
fields_sim = sk.fields(lines.species_keys)
sampled = fields_sim.sample(bundle.quadri_vectors[0], bundle.los_directions)
a_pts = np.asarray(bundle.quadri_vectors[0][..., 0])
v_pts = phys.c_kms * np.log(a_pts[:, :1] / a_pts)
fig, axs = plt.subplots(3, 3, figsize=(22, 12), sharex=True, sharey="row", layout="constrained")
for k in range(3):
    for j, l in enumerate(lines.lines):
        sty = MS["default"] if j == 0 else MS["matched"]
        axs[0, k].semilogy(v_pts[k], np.asarray(sampled[l.species]["number_density"])[k] / H.a ** 3, color=EC[l.species], **sty)
        axs[2, k].plot(velocities, flux[j, k], color=EC[l.species], **sty, label=l.id)
    axs[1, k].semilogy(v_pts[k], np.asarray(sampled["HI"]["temperature"])[k], color=EC["temperature"])
    axs[0, k].set_title(f"skewer {k}: y = {points[k, 0, 1]:.0f}, z = {points[k, 0, 2]:.0f} cMpc", fontsize=18); axs[2, k].set_xlabel("v [km/s]")
axs[0, 0].set_ylabel(r"$n$ [cm$^{-3}$]"); axs[1, 0].set_ylabel("T [K]"); axs[2, 0].set_ylabel("F"); axs[2, 0].legend()
add_panel_letters(axs, size=16); fig.suptitle("Three skewers through TNG50-4: the gas (top) and the flux it causes (bottom)", fontsize=22); plt.show()
print(f"<F>: Lya {flux[0].mean():.3f}, Si III {flux[1].mean():.4f}")